# Stage 2 — Complete monthly Sentinel-5P bivariate pipeline with Carbon Mapper plume matching

This notebook reproduces the complete Stage 2 workflow used for the thesis bivariate analysis.

It does four things:

1. Builds monthly Sentinel-5P/TROPOMI CH₄ products in Google Earth Engine:
   - valid-observation count `Vc`
   - exceedance count `Ec`, where XCH₄ > 1900 ppb
   - optional monthly mean XCH₄
   - fixed-bin bivariate class raster
2. Exports the monthly GeoTIFF products to Google Drive.
3. Locally samples the monthly bivariate GeoTIFFs at Carbon Mapper plume locations according to each plume timestamp.
4. Summarizes plume counts, reported emission-rate sums, instruments, and sectors by bivariate class.

The notebook is intentionally written as a reproducible research pipeline. It does not require hard-coded Earth Engine assets from the original development scripts.

**Important interpretation:** this workflow does not validate individual Carbon Mapper plumes with Sentinel-5P. It links plume observations to their monthly Sentinel-5P concentration context.

## 0. Install dependencies

Run this cell in Colab if the required geospatial packages are not already available.

In [ ]:
# Colab only. You can skip this cell if these packages are already installed.
!pip -q install earthengine-api geemap rasterio geopandas shapely pyproj matplotlib pandas numpy tqdm

## 1. Imports and configuration

Edit only the paths in this section. No API tokens are stored in the notebook.

In [ ]:
import os
import re
import glob
from pathlib import Path
from datetime import date

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import rasterio
from rasterio.warp import transform as rio_transform

# Optional Colab Drive mount
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

In [ ]:
# =========================
# USER CONFIGURATION
# =========================

# Earth Engine project. Use your project id if needed, otherwise keep None.
EE_PROJECT = None  # e.g. "global-booster-421311"

# Study period used in the thesis Stage 2 bivariate workflow.
START_YEAR = 2023
END_YEAR   = 2025

# Sentinel-5P methane band and exceedance threshold.
S5P_COLLECTION = "COPERNICUS/S5P/OFFL/L3_CH4"
S5P_BAND = "CH4_column_volume_mixing_ratio_dry_air_bias_corrected"
XCH4_THRESHOLD_PPB = 1900

# Output folder in Google Drive for Earth Engine exports.
# Earth Engine exports to Drive/MyDrive/<GEE_DRIVE_FOLDER> by default.
GEE_DRIVE_FOLDER = "stage2_bivariate_exports"

# Local folder where exported GeoTIFFs are available after Drive sync/mount.
# In Colab, exported files usually appear here after they complete in the Tasks tab.
LOCAL_RASTER_DIR = Path(f"/content/drive/MyDrive/{GEE_DRIVE_FOLDER}")

# Carbon Mapper plume CSV from Stage 0 / catalogue export.
# Replace with your own file path.
PLUME_CSV = Path("/content/drive/MyDrive/path_to_your_carbon_mapper_plumes.csv")

# Output folder for sampled plume table and figures.
OUT_DIR = Path("/content/drive/MyDrive/stage2_bivariate_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# If True, Earth Engine export tasks are submitted. Keep False until you are ready.
RUN_GEE_EXPORTS = False

# If True, local code will rebuild bivariate GeoTIFFs from exported valid/exceedance rasters.
# This is useful if you exported Vc and Ec but not the bivariate raster directly.
REBUILD_BIVAR_LOCALLY = True

print("LOCAL_RASTER_DIR:", LOCAL_RASTER_DIR)
print("OUT_DIR:", OUT_DIR)

## 2. Earth Engine initialization

This cell authenticates and initializes Earth Engine. If you use a named Google Cloud project, set `EE_PROJECT` above.

In [ ]:
import ee

try:
    if EE_PROJECT:
        ee.Initialize(project=EE_PROJECT)
    else:
        ee.Initialize()
except Exception:
    ee.Authenticate()
    if EE_PROJECT:
        ee.Initialize(project=EE_PROJECT)
    else:
        ee.Initialize()

print("Earth Engine initialized.")

## 3. Define CONUS study area and export grid

The thesis Stage 2 domain is the conterminous United States. We use TIGER state boundaries for the analysis region and a fixed 0.01° EPSG:4326 export grid for reproducible monthly rasters.

In [ ]:
# CONUS state abbreviations: all US states except Alaska and Hawaii.
CONUS_STATE_ABBRS = [
    'AL','AZ','AR','CA','CO','CT','DE','FL','GA','ID','IL','IN','IA','KS','KY','LA',
    'ME','MD','MA','MI','MN','MS','MO','MT','NE','NV','NH','NJ','NM','NY','NC','ND',
    'OH','OK','OR','PA','RI','SC','SD','TN','TX','UT','VT','VA','WA','WV','WI','WY'
]

states = ee.FeatureCollection("TIGER/2018/States").filter(
    ee.Filter.inList('STUSPS', CONUS_STATE_ABBRS)
)
CONUS = states.geometry()

# Export grid: 0.01 degree cells over an approximate CONUS bounding box.
# This matches the thesis description of a regular 0.01° monthly grid.
EXPORT_REGION = ee.Geometry.Rectangle([-125.0, 24.0, -66.0, 50.0], proj='EPSG:4326', geodesic=False)
CRS = 'EPSG:4326'
CRS_TRANSFORM = [0.01, 0, -125.0, 0, -0.01, 50.0]

print("CONUS and 0.01° export grid configured.")

## 4. Earth Engine functions: monthly Vc, Ec, mean XCH₄, and bivariate code

Definitions:

- `Vc`: valid-observation count per monthly grid cell.
- `Ec`: exceedance count per monthly grid cell, where valid XCH₄ > 1900 ppb.
- `NoEx`: valid support but zero exceedances.
- `NoData`: no valid support.

Fixed bins:

- class 1 = count 1–6
- class 2 = count 7–12
- class 3 = count 13–18
- class 4 = count ≥19

Bivariate codes:

- `1–16`: Vc1–Vc4 crossed with Ec1–Ec4
- `21–24`: NoEx classes for Vc1–Vc4 and Ec = 0
- `0`: NoData

In [ ]:
def month_start_end(year, month):
    start = ee.Date.fromYMD(int(year), int(month), 1)
    end = start.advance(1, 'month')
    return start, end


def s5p_month_collection(year, month):
    start, end = month_start_end(year, month)
    return (ee.ImageCollection(S5P_COLLECTION)
            .filterDate(start, end)
            .filterBounds(CONUS)
            .select(S5P_BAND))


def count_class_ee(count_img):
    """Return fixed count class 0, 1, 2, 3, 4 for an Earth Engine image."""
    count_img = ee.Image(count_img)
    cls = ee.Image(0).toInt16()
    cls = cls.where(count_img.gte(1).And(count_img.lte(6)), 1)
    cls = cls.where(count_img.gte(7).And(count_img.lte(12)), 2)
    cls = cls.where(count_img.gte(13).And(count_img.lte(18)), 3)
    cls = cls.where(count_img.gte(19), 4)
    return cls.toInt16()


def monthly_products_ee(year, month):
    """Build monthly valid count, exceedance count, mean XCH4, and bivariate class image."""
    col = s5p_month_collection(year, month)

    valid_count = col.count().unmask(0).clip(CONUS).rename('valid_count').toInt16()

    exceed_count = (col.map(lambda img: img.gt(XCH4_THRESHOLD_PPB).rename('exceed'))
                    .sum()
                    .unmask(0)
                    .clip(CONUS)
                    .rename('exceedance_count')
                    .toInt16())

    mean_xch4 = col.mean().clip(CONUS).rename('mean_xch4').toFloat()

    vc = count_class_ee(valid_count)
    ec = count_class_ee(exceed_count)

    # Codes 1–16 for Vc/Ec crossed classes.
    cross_code = vc.subtract(1).multiply(4).add(ec).toInt16()

    # Codes 21–24 for NoEx, i.e., Vc > 0 and Ec = 0.
    noex_code = vc.add(20).toInt16()

    bivar = (ee.Image(0).toInt16()
             .where(valid_count.gt(0).And(exceed_count.eq(0)), noex_code)
             .where(valid_count.gt(0).And(exceed_count.gt(0)), cross_code)
             .clip(CONUS)
             .rename('bivariate_class'))

    return valid_count, exceed_count, mean_xch4, bivar

print("Earth Engine product functions ready.")

## 5. Submit Earth Engine export tasks

Set `RUN_GEE_EXPORTS = True` in the configuration cell, then run this cell. The tasks will appear in the Earth Engine Tasks tab / Colab task list and export GeoTIFFs to your Google Drive folder.

You only need to do this once unless you change the threshold, period, domain, or grid.

In [ ]:
def export_image_to_drive(img, description, folder, file_prefix, dtype='int16'):
    task = ee.batch.Export.image.toDrive(
        image=img,
        description=description,
        folder=folder,
        fileNamePrefix=file_prefix,
        region=EXPORT_REGION,
        crs=CRS,
        crsTransform=CRS_TRANSFORM,
        maxPixels=1e13,
        fileFormat='GeoTIFF'
    )
    task.start()
    return task


submitted = []
if RUN_GEE_EXPORTS:
    for year in range(START_YEAR, END_YEAR + 1):
        for month in range(1, 13):
            ym = f"{year}_{month:02d}"
            valid_count, exceed_count, mean_xch4, bivar = monthly_products_ee(year, month)

            products = [
                (valid_count, f"S5P_N_valid_{ym}_CONUS", f"S5P_N_valid_{ym}_CONUS"),
                (exceed_count, f"S5P_N_exceed_{ym}_CONUS", f"S5P_N_exceed_{ym}_CONUS"),
                (mean_xch4, f"S5P_XCH4_mean_{ym}_CONUS", f"S5P_XCH4_mean_{ym}_CONUS"),
                (bivar, f"bivariate_custombins_{ym}_CONUS_K4plus", f"bivariate_custombins_{ym}_CONUS_K4plus"),
            ]
            for img, desc, prefix in products:
                submitted.append(export_image_to_drive(img, desc, GEE_DRIVE_FOLDER, prefix))
                print("submitted:", desc)
else:
    print("RUN_GEE_EXPORTS is False. No tasks submitted.")

print("Number of tasks submitted:", len(submitted))

## 6. Local fixed-bin bivariate builder from exported Vc/Ec rasters

This section makes the notebook robust: even if you export only `S5P_N_valid_*.tif` and `S5P_N_exceed_*.tif`, the fixed-bin bivariate GeoTIFFs can be reconstructed locally.

In [ ]:
def classify_count_np(arr):
    """Classify count array into 0,1,2,3,4 using bins 1–6, 7–12, 13–18, >=19."""
    arr = np.asarray(arr)
    out = np.zeros(arr.shape, dtype=np.uint8)
    out[(arr >= 1) & (arr <= 6)] = 1
    out[(arr >= 7) & (arr <= 12)] = 2
    out[(arr >= 13) & (arr <= 18)] = 3
    out[arr >= 19] = 4
    return out


def build_bivar_np(valid_count, exceed_count):
    """Build bivariate code array from valid and exceedance count arrays."""
    valid_count = np.asarray(valid_count)
    exceed_count = np.asarray(exceed_count)

    vc = classify_count_np(valid_count)
    ec = classify_count_np(exceed_count)

    out = np.zeros(valid_count.shape, dtype=np.uint8)  # 0 = NoData

    noex = (valid_count > 0) & (exceed_count == 0)
    cross = (valid_count > 0) & (exceed_count > 0)

    out[noex] = 20 + vc[noex]               # 21–24
    out[cross] = (vc[cross] - 1) * 4 + ec[cross]  # 1–16

    return out


def find_raster(pattern):
    matches = sorted(glob.glob(str(LOCAL_RASTER_DIR / pattern)))
    return Path(matches[0]) if matches else None


def rebuild_bivariate_for_month(year, month, overwrite=False):
    ym = f"{year}_{month:02d}"
    valid_path = find_raster(f"S5P_N_valid_{ym}_CONUS*.tif")
    exceed_path = find_raster(f"S5P_N_exceed_{ym}_CONUS*.tif")
    out_path = LOCAL_RASTER_DIR / f"bivariate_custombins_{ym}_CONUS_K4plus.tif"

    if out_path.exists() and not overwrite:
        return out_path

    if valid_path is None or exceed_path is None:
        print(f"Missing valid/exceedance raster for {ym}")
        return None

    with rasterio.open(valid_path) as src_v, rasterio.open(exceed_path) as src_e:
        valid = src_v.read(1)
        exceed = src_e.read(1)
        if valid.shape != exceed.shape:
            raise ValueError(f"Shape mismatch for {ym}: {valid.shape} vs {exceed.shape}")
        if src_v.transform != src_e.transform or src_v.crs != src_e.crs:
            raise ValueError(f"Grid/CRS mismatch for {ym}. Align rasters before building bivariate map.")
        bivar = build_bivar_np(valid, exceed)
        profile = src_v.profile.copy()

    profile.update(dtype='uint8', count=1, nodata=0, compress='lzw')
    with rasterio.open(out_path, 'w', **profile) as dst:
        dst.write(bivar, 1)

    return out_path


if REBUILD_BIVAR_LOCALLY:
    made = []
    for year in range(START_YEAR, END_YEAR + 1):
        for month in range(1, 13):
            p = rebuild_bivariate_for_month(year, month, overwrite=False)
            if p is not None:
                made.append(p)
    print(f"Available/rebuilt bivariate rasters: {len(made)}")
    print("Example:", made[:3])
else:
    print("REBUILD_BIVAR_LOCALLY is False.")

## 7. Quick QA of bivariate class codes

This checks that the rasters contain only the expected codes:

- `0` = NoData
- `1–16` = Vc/Ec crossed classes
- `21–24` = NoEx classes

In [ ]:
expected_codes = set([0] + list(range(1, 17)) + list(range(21, 25)))
qa_rows = []
for path in sorted(LOCAL_RASTER_DIR.glob("bivariate_custombins_*_CONUS_K4plus*.tif")):
    with rasterio.open(path) as src:
        arr = src.read(1)
    vals, counts = np.unique(arr, return_counts=True)
    bad = sorted(set(vals.tolist()) - expected_codes)
    qa_rows.append({
        "file": path.name,
        "n_pixels": int(arr.size),
        "nonzero_pixels": int((arr > 0).sum()),
        "codes": ",".join(map(str, vals.tolist())),
        "unexpected_codes": ",".join(map(str, bad)) if bad else ""
    })
qa_df = pd.DataFrame(qa_rows)
display(qa_df.head())
qa_df.to_csv(OUT_DIR / "bivariate_raster_code_QA.csv", index=False)
print("Saved:", OUT_DIR / "bivariate_raster_code_QA.csv")

## 8. Load and standardize Carbon Mapper plume catalogue

The plume table must include longitude, latitude, timestamp, gas, and ideally `emission_auto`, source sector, and instrument/platform columns.

The code below tries common Carbon Mapper column names and keeps the pipeline flexible.

In [ ]:
def first_existing(columns, candidates):
    for c in candidates:
        if c in columns:
            return c
    return None


def normalize_gas_series(s):
    return (s.astype(str).str.upper()
            .str.replace('₄', '4', regex=False)
            .str.replace('₂', '2', regex=False)
            .replace({'METHANE': 'CH4', 'CARBON DIOXIDE': 'CO2', 'CARBONDIOXIDE': 'CO2'}))


plumes = pd.read_csv(PLUME_CSV)
print("Loaded plume rows:", len(plumes))
print("Columns:", list(plumes.columns))

lon_col = first_existing(plumes.columns, ['plume_longitude', 'longitude', 'lon', 'source_longitude'])
lat_col = first_existing(plumes.columns, ['plume_latitude', 'latitude', 'lat', 'source_latitude'])
time_col = first_existing(plumes.columns, ['datetime', 'acquired', 'timestamp', 'time', 'date'])
gas_col = first_existing(plumes.columns, ['gas', 'gas_norm', 'species'])
emis_col = first_existing(plumes.columns, ['emission_auto', 'emission_rate', 'source_rate', 'emissions'])
sector_col = first_existing(plumes.columns, ['ipcc_sector', 'sector', 'source_sector', 'sector_name'])
instr_col = first_existing(plumes.columns, ['instrument', 'platform', 'instrument_platform'])

required = {'longitude': lon_col, 'latitude': lat_col, 'time': time_col}
missing = [k for k, v in required.items() if v is None]
if missing:
    raise ValueError(f"Missing required columns: {missing}. Available columns: {list(plumes.columns)}")

plumes = plumes.copy()
plumes['longitude'] = pd.to_numeric(plumes[lon_col], errors='coerce')
plumes['latitude'] = pd.to_numeric(plumes[lat_col], errors='coerce')
plumes['datetime_parsed'] = pd.to_datetime(plumes[time_col], errors='coerce', utc=True)
plumes['year'] = plumes['datetime_parsed'].dt.year
plumes['month'] = plumes['datetime_parsed'].dt.month
plumes['year_month'] = plumes['datetime_parsed'].dt.strftime('%Y_%m')

if gas_col:
    plumes['gas_norm'] = normalize_gas_series(plumes[gas_col])
else:
    plumes['gas_norm'] = 'CH4'

# Stage 2 thesis analysis is methane-only.
plumes = plumes[(plumes['gas_norm'] == 'CH4') & plumes['longitude'].notna() & plumes['latitude'].notna() & plumes['datetime_parsed'].notna()].copy()
plumes = plumes[(plumes['year'] >= START_YEAR) & (plumes['year'] <= END_YEAR)].copy()

print("CH4 plume rows in study period:", len(plumes))
display(plumes.head())

## 9. Sample monthly bivariate rasters at plume locations

For each Carbon Mapper plume, this section selects the bivariate raster corresponding to the plume acquisition month and samples the raster at the plume longitude/latitude.

In [ ]:
def bivar_raster_for_year_month(year, month):
    ym = f"{int(year)}_{int(month):02d}"
    matches = sorted(LOCAL_RASTER_DIR.glob(f"bivariate_custombins_{ym}_CONUS_K4plus*.tif"))
    if not matches:
        return None
    return matches[0]


def decode_bivar_code(code):
    if pd.isna(code):
        return pd.Series({
            'bivar_group': 'NoData', 'vc_class': np.nan, 'ec_class': np.nan,
            'is_noex': False, 'is_nodata': True, 'class_label': 'NoData'
        })
    code = int(code)
    if code == 0:
        return pd.Series({'bivar_group': 'NoData', 'vc_class': np.nan, 'ec_class': np.nan,
                          'is_noex': False, 'is_nodata': True, 'class_label': 'NoData'})
    if 21 <= code <= 24:
        vc = code - 20
        return pd.Series({'bivar_group': 'NoEx', 'vc_class': vc, 'ec_class': 0,
                          'is_noex': True, 'is_nodata': False, 'class_label': f'Vc{vc}-NoEx'})
    if 1 <= code <= 16:
        vc = ((code - 1) // 4) + 1
        ec = ((code - 1) % 4) + 1
        return pd.Series({'bivar_group': 'Exceedance', 'vc_class': vc, 'ec_class': ec,
                          'is_noex': False, 'is_nodata': False, 'class_label': f'Vc{vc}-Ec{ec}'})
    return pd.Series({'bivar_group': 'Unexpected', 'vc_class': np.nan, 'ec_class': np.nan,
                      'is_noex': False, 'is_nodata': False, 'class_label': f'Unexpected-{code}'})


def sample_points_from_raster(df_month, raster_path):
    with rasterio.open(raster_path) as src:
        xs = df_month['longitude'].to_numpy(dtype=float)
        ys = df_month['latitude'].to_numpy(dtype=float)

        # Transform coordinates to raster CRS if needed.
        if src.crs and src.crs.to_string() not in ['EPSG:4326', 'OGC:CRS84']:
            xs, ys = rio_transform('EPSG:4326', src.crs, xs.tolist(), ys.tolist())

        coords = list(zip(xs, ys))
        vals = []
        for val in src.sample(coords):
            vals.append(int(val[0]) if len(val) else 0)
    return vals


sampled_parts = []
missing_months = []

for (year, month), dfm in tqdm(plumes.groupby(['year', 'month']), desc='Sampling monthly rasters'):
    path = bivar_raster_for_year_month(year, month)
    temp = dfm.copy()
    if path is None:
        temp['bivar_code'] = np.nan
        missing_months.append(f"{int(year)}_{int(month):02d}")
    else:
        temp['bivar_raster'] = path.name
        temp['bivar_code'] = sample_points_from_raster(temp, path)
    sampled_parts.append(temp)

sampled = pd.concat(sampled_parts, ignore_index=True) if sampled_parts else pd.DataFrame()

# Decode classes
decoded = sampled['bivar_code'].apply(decode_bivar_code)
sampled = pd.concat([sampled.reset_index(drop=True), decoded.reset_index(drop=True)], axis=1)

print("Sampled plume rows:", len(sampled))
print("Missing bivariate months:", sorted(set(missing_months)))
display(sampled[['datetime_parsed','longitude','latitude','bivar_code','class_label','bivar_group']].head())

sampled_path = OUT_DIR / "carbon_mapper_plumes_sampled_by_monthly_bivariate_class.csv"
sampled.to_csv(sampled_path, index=False)
print("Saved:", sampled_path)

## 10. Main Stage 2 summary: classified, NoData, NoEx, non-NoEx

This is the central bivariate matching output. It reports how many plumes were assigned to a classified monthly Sentinel-5P class and how many classified plume emissions fell in `NoEx` versus non-`NoEx` classes.

In [ ]:
classified = sampled[~sampled['is_nodata'] & sampled['bivar_code'].notna()].copy()
nodata = sampled[sampled['is_nodata'] | sampled['bivar_code'].isna()].copy()

summary_rows = []
summary_rows.append({'group': 'All plume records', 'plumes': len(sampled)})
summary_rows.append({'group': 'Classified bivariate pixels', 'plumes': len(classified)})
summary_rows.append({'group': 'NoData / missing raster support', 'plumes': len(nodata)})

summary = pd.DataFrame(summary_rows)
display(summary)

if emis_col and emis_col in sampled.columns:
    sampled['emission_value'] = pd.to_numeric(sampled[emis_col], errors='coerce')
else:
    sampled['emission_value'] = np.nan

if sampled['emission_value'].notna().any():
    g_noex = classified[classified['is_noex']]
    g_ex = classified[~classified['is_noex']]
    classified_emis_sum = classified['emission_value'].sum(skipna=True)
    noex_emis_sum = g_noex['emission_value'].sum(skipna=True)
    ex_emis_sum = g_ex['emission_value'].sum(skipna=True)

    noex_summary = pd.DataFrame([
        {'group': 'NoEx', 'plumes': len(g_noex), 'emission_sum': noex_emis_sum,
         'share_of_classified_emission_%': 100 * noex_emis_sum / classified_emis_sum if classified_emis_sum else np.nan},
        {'group': 'All non-NoEx classes', 'plumes': len(g_ex), 'emission_sum': ex_emis_sum,
         'share_of_classified_emission_%': 100 * ex_emis_sum / classified_emis_sum if classified_emis_sum else np.nan},
        {'group': 'NoData', 'plumes': len(nodata), 'emission_sum': nodata['emission_value'].sum(skipna=True),
         'share_of_classified_emission_%': np.nan},
    ])
    noex_summary['emission_sum'] = noex_summary['emission_sum'].round(3)
    noex_summary['share_of_classified_emission_%'] = noex_summary['share_of_classified_emission_%'].round(2)
    display(noex_summary)
    noex_summary.to_csv(OUT_DIR / "noex_vs_non_noex_emission_summary.csv", index=False)
else:
    print("No emission column found or all values are NaN; emission-weighted summary skipped.")

## 11. Heatmaps of plume counts and emission sums by bivariate class

Rows are valid-count classes `Vc1–Vc4`; columns are `NoEx` and exceedance classes `Ec1–Ec4`.

In [ ]:
def bivar_matrix(df, value_col=None):
    mat = pd.DataFrame(0.0, index=[1,2,3,4], columns=['NoEx', 'Ec1', 'Ec2', 'Ec3', 'Ec4'])
    d = df[~df['is_nodata']].copy()
    for vc in [1,2,3,4]:
        # NoEx
        mask = (d['vc_class'] == vc) & (d['ec_class'] == 0)
        mat.loc[vc, 'NoEx'] = d.loc[mask, value_col].sum() if value_col else mask.sum()
        for ec in [1,2,3,4]:
            mask = (d['vc_class'] == vc) & (d['ec_class'] == ec)
            mat.loc[vc, f'Ec{ec}'] = d.loc[mask, value_col].sum() if value_col else mask.sum()
    return mat

count_mat = bivar_matrix(sampled)
display(count_mat.astype(int))

plt.figure(figsize=(7,5))
plt.imshow(count_mat.values, aspect='auto')
plt.xticks(range(count_mat.shape[1]), count_mat.columns)
plt.yticks(range(count_mat.shape[0]), [f'Vc{v}' for v in count_mat.index])
plt.xlabel('Exceedance class Ec; NoEx = Ec=0')
plt.ylabel('Valid-count class Vc')
plt.title('Carbon Mapper plume counts by monthly bivariate class')
for i in range(count_mat.shape[0]):
    for j in range(count_mat.shape[1]):
        plt.text(j, i, int(count_mat.values[i,j]), ha='center', va='center')
plt.colorbar(label='Plume count')
plt.tight_layout()
plt.savefig(OUT_DIR / 'heatmap_plume_counts_by_bivariate_class.png', dpi=200)
plt.show()

if sampled['emission_value'].notna().any():
    emis_mat = bivar_matrix(sampled, value_col='emission_value')
    display(emis_mat.round(1))
    plt.figure(figsize=(7,5))
    plt.imshow(emis_mat.values, aspect='auto')
    plt.xticks(range(emis_mat.shape[1]), emis_mat.columns)
    plt.yticks(range(emis_mat.shape[0]), [f'Vc{v}' for v in emis_mat.index])
    plt.xlabel('Exceedance class Ec; NoEx = Ec=0')
    plt.ylabel('Valid-count class Vc')
    plt.title('Emission-rate sums by monthly bivariate class')
    for i in range(emis_mat.shape[0]):
        for j in range(emis_mat.shape[1]):
            plt.text(j, i, f"{emis_mat.values[i,j]/1000:.1f}k", ha='center', va='center')
    plt.colorbar(label='Emission-rate sum')
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'heatmap_emission_sums_by_bivariate_class.png', dpi=200)
    plt.show()

## 12. Sector and instrument summaries by bivariate class

These summaries reproduce the interpretation layer of Stage 2: which sectors and instruments dominate each Sentinel-5P class.

In [ ]:
# Sector summary
if sector_col and sector_col in sampled.columns:
    sector_summary = (sampled[~sampled['is_nodata']]
                      .groupby(['class_label', sector_col], dropna=False)
                      .size()
                      .reset_index(name='plume_count')
                      .sort_values(['class_label', 'plume_count'], ascending=[True, False]))
    display(sector_summary.head(20))
    sector_summary.to_csv(OUT_DIR / 'sector_counts_by_bivariate_class.csv', index=False)

    # Stacked bar for most populated classes and most common sectors.
    top_classes = sampled[~sampled['is_nodata']]['class_label'].value_counts().head(12).index.tolist()
    top_sectors = sampled[sector_col].astype(str).value_counts().head(8).index.tolist()
    plot_df = sampled[sampled['class_label'].isin(top_classes)].copy()
    plot_df[sector_col] = plot_df[sector_col].astype(str).where(plot_df[sector_col].astype(str).isin(top_sectors), 'Other')
    pivot = pd.crosstab(plot_df['class_label'], plot_df[sector_col]).loc[top_classes]
    pivot.plot(kind='bar', stacked=True, figsize=(12,5))
    plt.ylabel('Plume count')
    plt.xlabel('Bivariate class')
    plt.title('Sector composition within most populated bivariate classes')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'sector_composition_by_bivariate_class.png', dpi=200)
    plt.show()
else:
    print("No sector column found; sector summary skipped.")

# Instrument/platform summary
if instr_col and instr_col in sampled.columns:
    instrument_summary = (sampled[~sampled['is_nodata']]
                          .groupby(['class_label', instr_col], dropna=False)
                          .size()
                          .reset_index(name='plume_count')
                          .sort_values(['class_label', 'plume_count'], ascending=[True, False]))
    display(instrument_summary.head(20))
    instrument_summary.to_csv(OUT_DIR / 'instrument_counts_by_bivariate_class.csv', index=False)
else:
    print("No instrument/platform column found; instrument summary skipped.")

## 13. Outputs created by this notebook

Expected output files:

- monthly GeoTIFFs exported from Earth Engine:
  - `S5P_N_valid_YYYY_MM_CONUS.tif`
  - `S5P_N_exceed_YYYY_MM_CONUS.tif`
  - `S5P_XCH4_mean_YYYY_MM_CONUS.tif`
  - `bivariate_custombins_YYYY_MM_CONUS_K4plus.tif`
- local sampled plume table:
  - `carbon_mapper_plumes_sampled_by_monthly_bivariate_class.csv`
- summary tables:
  - `bivariate_raster_code_QA.csv`
  - `noex_vs_non_noex_emission_summary.csv`
  - `sector_counts_by_bivariate_class.csv`
  - `instrument_counts_by_bivariate_class.csv`
- figures:
  - `heatmap_plume_counts_by_bivariate_class.png`
  - `heatmap_emission_sums_by_bivariate_class.png`
  - `sector_composition_by_bivariate_class.png`

This notebook is the complete Stage 2 pipeline: **GEE monthly rasters → fixed-bin bivariate maps → Carbon Mapper plume matching → class summaries and figures**.